# 🐝 BeeOne Purchase Order Extractor — VLM v7

Extract structured data from French purchase orders (bons de commande) using:
- **Qwen 2.5 VL** — Vision-Language Model that SEES the document image directly
- **No OCR needed** — the model reads and understands layout at the same time
- **4-bit quantization** — runs on Colab's free T4 GPU (~6GB VRAM)

### VLM Architecture (upgraded from EasyOCR + text LLM):
```
Old: PDF → Image → EasyOCR → Flat Text → Qwen 2.5 → JSON  (2 models, OCR errors)
New: PDF → Image → Qwen 2.5 VL → JSON                           (1 model, sees layout)
```

---

### How to use:
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Run all cells in order (**Runtime → Run all**)
3. Upload your document in the Gradio interface at the bottom
4. Get structured JSON output

### Customize extraction:
- Edit **`output_schema.json`** to add/remove/rename output fields
- Edit **`extraction_rules.yaml`** to improve field detection
- Edit **`prompt_template.txt`** to change the extraction prompt
- No Python changes needed!


## Step 1: Check GPU

Make sure you're using a **T4 GPU** (free) or better. Go to **Runtime → Change runtime type** if needed.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {vram_gb:.1f} GB")
    print(f"✅ PyTorch: {torch.__version__}")
    print(f"\n  Recommended models:")
    if vram_gb >= 14:
        print(f"    • Qwen/Qwen2.5-VL-7B-Instruct  (best quality, ~5GB VRAM)")
        print(f"    • Qwen/Qwen2.5-VL-72B-Instruct (best quality, ~10GB VRAM)")
    else:
        print(f"    • Qwen/Qwen2.5-VL-7B-Instruct  (best quality, ~5GB VRAM)")
    print(f"    • Qwen/Qwen2.5-VL-3B-Instruct  (faster, ~2.5GB VRAM)")
    print(f"    • Qwen/Qwen2.5-VL-3B-Instruct (fastest, ~1.5GB VRAM)")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
    print("   The model CAN run on CPU but will be VERY slow (5-10 minutes per document).")

## Step 2: Install Dependencies

This installs Transformers, Qwen VL utils, Gradio, and all required libraries.
Takes ~1-2 minutes.


In [ ]:
# VLM dependencies — no EasyOCR needed, the model sees the image directly
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes qwen-vl-utils "numpy>=2.0.0" "protobuf>=5.28.0,<6.0.0" --no-cache-dir

!pip install -q PyMuPDF Pillow pyyaml pydantic httpx gradio reportlab sentencepiece "opencv-python-headless>=4.8.0" --no-cache-dir

print("✅ All dependencies installed")


## Step 3: Create Config Files

These 3 files control what gets extracted and how.
**Edit them anytime** — no Python changes needed.

| File | Purpose |
|------|----------|
| `output_schema.json` | Defines the JSON output structure |
| `extraction_rules.yaml` | Field descriptions, keywords, model settings |
| `prompt_template.txt` | Base prompt template |

In [ ]:
import os

CONFIG_DIR = "/content/beeone_config"
os.makedirs(CONFIG_DIR, exist_ok=True)
print(f"✅ Config directory: {CONFIG_DIR}")

In [ ]:
# ==========================================================================
# output_schema.json — Edit this to customize the JSON output structure
# ==========================================================================
import json

output_schema = {
    "document_type": "Purchase Order",
    "language": "auto-detected (French or English)",
    "output": {
        "seller": {
            "name": "string — Legal name of the seller/vendor company",
            "address": "string — Full street address",
            "postal_code": "string — Postal / ZIP code",
            "city": "string — City name",
            "country": "string — Country",
            "siret": "string — SIRET number (14 digits)",
            "phone": "string — Phone number",
            "email": "string — Email address",
            "vat_number": "string — TVA intracommunautaire number",
            "iban": "string — IBAN"
        },
        "buyer": {
            "company_name": "string — Buyer/client company name",
            "address": "string — Full street address",
            "postal_code": "string — Postal / ZIP code",
            "city": "string — City name",
            "country": "string — Country",
            "contact_name": "string — Contact person",
            "phone": "string — Phone number",
            "email": "string — Email address"
        },
        "additional_information": "string — Notes, remarks, or observations at bottom of document (Informations complementaires / Notes). Use null if absent.",
        "order": {
            "order_number": "string — Purchase order reference number",
            "date": "string — Order date in YYYY-MM-DD format",
            "ferme": "string — Farm name, code, or identifier (the value next to the Ferme field, e.g. SPMS 115). Use null if not present on the document.",
            "payment_conditions": "string — Payment terms",
            "delivery_address": "string — Delivery address if different from buyer",
            "delivery_delay": {
                "value": "string — Numeric value of delivery delay",
                "unit": "string — Unit (jours, semaines)"
            },
            "items": [
                {
                    "reference": "string — Item reference / product code",
                    "designation": "string — Product name / description",
                    "quantity": "number — Quantity ordered",
                    "unit": "string — Unit of measurement",
                    "unit_price_ht": "number — Unit price before tax",
                    "total_line_ht": "number — Line total before tax",
                    "vat_rate": "number — VAT rate percentage",
                    "discount": "number — Discount on this line"
                }
            ],
            "totals": {
                "gross_amount_ht": "number — Subtotal before discounts",
                "discount": "number — Total discount amount",
                "net_amount_ht": "number — Net amount before tax",
                "vat_amount": "number — Total VAT amount",
                "vat_rate": "number — Applied VAT rate percentage",
                "net_amount_ttc": "number — Final total including tax (TTC)",
                "amount_in_words": "string — Amount in words"
            }
        }
    }
}

with open(f"{CONFIG_DIR}/output_schema.json", "w") as f:
    json.dump(output_schema, f, indent=2, ensure_ascii=False)

print("✅ output_schema.json created")
print(f"   Fields: seller ({len(output_schema['output']['seller'])}), "
      f"buyer ({len(output_schema['output']['buyer'])}), "
      f"order ({len(output_schema['output']['order'])})")

In [ ]:
# ==========================================================================
# extraction_rules.yaml — Edit field descriptions and model settings
# ==========================================================================
import yaml

extraction_rules = {
    "model": {
        # Change this to switch models:
        # "Qwen/Qwen2.5-VL-7B-Instruct"    — RECOMMENDED VLM for T4 GPU (~6GB VRAM)
        # "Qwen/Qwen2.5-VL-7B-Instruct (larger)"   — better quality but may OOM on T4 (14GB VRAM)
        # "Qwen/Qwen2.5-VL-3B-Instruct"  — smaller/faster (~4GB VRAM)
        "name": "Qwen/Qwen2.5-VL-7B-Instruct",
        "temperature": 0.1,
        "top_p": 0.9,
        "num_predict": 4096,
    },
    "global_rules": [
        "Extract ONLY information explicitly present in the document.",
        "Never invent, guess, or fabricate values.",
        "Use null for unknown values.",
        "Monetary amounts must be numbers (e.g. 1250.50, not '1 250,50 €').",
        "Dates must be ISO format YYYY-MM-DD.",
        "Extract ALL line items, in order.",
        "Preserve original French text.",
        "Convert European numbers: '1 250,50' → 1250.50.",
    ],
    "fields": {
        "seller.name": {
            "description": "The vendor/seller company name",
            "location": "Top-left or top-right, often near a logo",
            "keywords": ["Vendeur", "Fournisseur", "Société", "Entreprise"],
            "avoid": ["Ville:", "Adresse:", "Tél:"]
        },
        "seller.siret": {
            "description": "French company registration number (14 digits)",
            "keywords": ["SIRET", "N° SIRET"]
        },
        "seller.vat_number": {
            "description": "VAT number for intra-community transactions",
            "keywords": ["TVA", "N° TVA", "TVA intracommunautaire"]
        },
        "buyer.company_name": {
            "description": "The purchasing company",
            "keywords": ["Client", "Acheteur", "Commanditaire", "Destinataire"]
        },
        "order.order_number": {
            "description": "Unique purchase order reference number",
            "keywords": ["N° de commande", "Référence", "N° BC", "Bon de commande N°", "NUMÉRO DE BON DE COMMANDE"],
            "examples": ["BC-2025-001", "F2018-2016"]
        },
        "order.date": {
            "description": "Order issue date in YYYY-MM-DD",
            "keywords": ["Date", "Date de commande", "Le"]
        },
        "order.ferme": {
            "description": "Farm name, code, or facility identifier written next to Ferme:",
            "keywords": ["Ferme", "Fermé", "Commande ferme"],
            "notes": "Extract the VALUE next to Ferme (e.g. SPMS 115, Ferme 1, etc.), not the word Ferme itself."
        },
        "order.delivery_delay": {
            "description": "Expected delivery timeframe",
            "keywords": ["Délai de livraison", "Livraison prévue", "Livraison sous"]
        },
        "order.payment_conditions": {
            "description": "Payment terms",
            "keywords": ["Conditions de paiement", "Paiement", "Échéance"]
        },
        "order.items": {
            "description": "All line items in the order table",
            "keywords": ["Réf", "Désignation", "Qté", "Prix unitaire"],
            "notes": "Extract EVERY row. The array must match the document exactly."
        },
        "order.totals.net_amount_ttc": {
            "description": "Grand total including all taxes (TTC)",
            "keywords": ["Total TTC", "Montant TTC", "Net à payer", "Total général"]
        }
    }
}

with open(f"{CONFIG_DIR}/extraction_rules.yaml", "w") as f:
    yaml.dump(extraction_rules, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print("✅ extraction_rules.yaml created")
print(f"   Model: {extraction_rules['model']['name']}")
print(f"   Fields configured: {len(extraction_rules['fields'])}")

In [ ]:
# ======================================================================
# Bilingual Prompts (French + English)
# ======================================================================

PROMPT_FR = """SYSTEM PROMPT:
You are a highly precise document data extraction assistant specialized in French business documents (purchase orders, invoices, delivery notes). You can SEE the document image directly \u2014 use your vision to read the text and understand the layout.

STRICT RULES:
1. Output ONLY valid JSON \u2014 no explanations, no markdown code fences, no commentary.
2. Every field in the schema must be present in your output. Use null for unknown values.
3. Monetary values must be numbers (e.g. 1250.50), not strings. Remove currency symbols.
4. Convert European number formatting: spaces as thousand separators and commas as decimal \u2192 standard format (e.g. "1 250,50" \u2192 1250.50).
5. Dates must be ISO format YYYY-MM-DD. Convert dd/mm/yyyy \u2192 yyyy-mm-dd.
6. Boolean fields: true or false (not "oui"/"non"). The "ferme" field is NOT a boolean — it is the farm name/code written next to "Ferme:" on the document.
7. The items array must contain ALL line items found in the document, in their original order.
8. Preserve original French text for description fields (do not translate).
9. Never invent, guess, or hallucinate values not present in the document.
10. If the same information appears multiple times, use the most complete/precise version.
11. For totals, prefer explicit "Total TTC" / "Net \u00e0 payer" values.
12. IMPORTANT: The SELLER is the company ISSUING the document (usually top of page, often near logo). The BUYER is the company RECEIVING the order (usually labeled "Client" or "Destinataire"). Do NOT confuse them.
13. TABLE COLUMNS: Look at the column HEADERS to identify what each number means. The columns may not be in standard order \u2014 use the header labels to determine which column is R\u00e9f, D\u00e9signation, Quantit\u00e9, Unit\u00e9, PU HT, Montant HT, etc.
14. Read numbers carefully from the image. If a number is partially obscured, make your best guess but mark uncertain values with null.
15. INFORMATIONS COMPLEMENTAIRES (MANDATORY): Look at the VERY BOTTOM of the document for an "Informations complementaires" or "Observations" section. The "additional_information" is a TOP-LEVEL field (between buyer and order, NOT inside totals). You MUST include this field in your output even if its value is null. Extract ALL text from this section.
16. TTC PRIORITY: Always use the EXPLICIT "Total Montant net TTC" or "Net a payer" value from the document, even if it does not match the calculated gross + VAT. The document is the source of truth.
17. FINAL CHECK: Before outputting, verify "additional_information" is present as a top-level field. If you forgot it, add it now.

FIELD EXTRACTION GUIDANCE:
{extraction_hints}

OUTPUT SCHEMA:
{output_schema}

Now look at this purchase order image carefully and extract all data into the JSON format above."""


PROMPT_EN = """SYSTEM PROMPT:
You are a highly precise document data extraction assistant specialized in business documents (purchase orders, invoices, delivery notes). You can SEE the document image directly \u2014 use your vision to read the text and understand the layout.

STRICT RULES:
1. Output ONLY valid JSON \u2014 no explanations, no markdown code fences, no commentary.
2. Every field in the schema must be present in your output. Use null for unknown values.
3. Monetary values must be numbers (e.g. 1250.50), not strings. Remove currency symbols.
4. Dates must be ISO format YYYY-MM-DD. Convert any date format to yyyy-mm-dd.
5. Boolean fields: true or false. The "ferme" field is NOT a boolean — it is the farm name/code written next to "Ferme:" on the document.
6. The items array must contain ALL line items found in the document, in their original order.
7. Preserve original text for description fields (do not translate).
8. Never invent, guess, or hallucinate values not present in the document.
9. For totals, prefer explicit "Grand Total" / "Total" values.
10. IMPORTANT: The SELLER is the company ISSUING the document (usually top of page, often near logo). The BUYER is the company RECEIVING the order (usually labeled "Customer", "Bill To", or "Ship To"). Do NOT confuse them.
11. TABLE COLUMNS: Look at the column HEADERS to identify what each number means. Use the header labels to determine which column is Reference, Description, Quantity, Unit, Unit Price, Total, etc.
12. Read numbers carefully from the image.
13. ADDITIONAL INFO (MANDATORY): Look at the VERY BOTTOM of the document for "Notes", "Remarks", "Additional Information", or "Observations" sections. The "additional_information" is a TOP-LEVEL field (between buyer and order, NOT inside totals). You MUST include this field even if its value is null.
14. FINAL CHECK: Before outputting, verify "additional_information" is present as a top-level field.
15. TTC PRIORITY: Always use the EXPLICIT total value written on the document, even if it does not match your calculation. The document is the source of truth.

FIELD EXTRACTION GUIDANCE:
{extraction_hints}

OUTPUT SCHEMA:
{output_schema}

Now look at this purchase order image carefully and extract all data into the JSON format above."""


# ======================================================================
# Language Detection
# ======================================================================

def detect_language(file_bytes, filename):
    """Detect if a document is French or English using quick text scan."""
    try:
        ext = os.path.splitext(filename)[1].lower()
        if ext == ".pdf":
            doc = fitz.open(stream=file_bytes, filetype="pdf")
            text = doc[0].get_text() if len(doc) > 0 else ""
            doc.close()
        else:
            img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
            text = ""
    except Exception:
        return "fr"
    
    french_markers = [
        "bon de commande", "fournisseur", "montant", "total ttc", "total ht",
        "siret", "tva", "facture", "societe", "adresse de livraison",
        "conditions de reglement", "delai de livraison", "ferme",
        "montant brut", "montant net", "informations complementaires",
        "numero de commande", "designation", "quantite", "unite",
        "net a payer", "dirhams", "dhs", "maroc",
    ]
    english_markers = [
        "purchase order", "supplier", "vendor", "subtotal", "grand total",
        "invoice", "shipping address", "payment terms", "delivery",
        "qty", "quantity", "unit price", "description", "item",
        "order number", "po number", "total amount", "due date",
        "bill to", "ship to", "tax id", "vat number",
        "company", "limited", "ltd", "inc", "llc",
    ]
    
    text_lower = text.lower()
    french_score = sum(2 for m in french_markers if m in text_lower)
    english_score = sum(2 for m in english_markers if m in text_lower)
    
    # French accented characters are a strong signal
    if any(c in text for c in ["\u00e9", "\u00e8", "\u00ea", "\u00e0", "\u00fb"]):
        french_score += 3
    
    return "fr" if french_score >= english_score else "en"


def get_prompt_for_language(lang):
    """Return the appropriate prompt for detected language."""
    return PROMPT_FR if lang == "fr" else PROMPT_EN


prompt_template = get_prompt_for_language("fr")  # default, will be overridden at extraction time

with open(f"{CONFIG_DIR}/prompt_template.txt", "w") as f:
    f.write(prompt_template)

print("\u2705 Bilingual prompts ready (French + English)")
print("   Language will be auto-detected from each document")


## Step 4: Download & Load the Vision-Language Model (4-bit Quantized)

This downloads **Qwen 2.5 VL 7B** (Vision-Language) and loads it in **4-bit quantization** (~6GB VRAM).
The VL model can **see images** and understand document layout — no separate OCR needed!
Fits on T4 GPU (15GB) with plenty of room.
Takes ~3-5 minutes on first run (download).


In [ ]:
import json
import time
import logging
import gc
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")

# Reduce CUDA memory fragmentation during model loading
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load model name from config
with open(f"{CONFIG_DIR}/extraction_rules.yaml") as f:
    rules = yaml.safe_load(f)

MODEL_NAME = rules["model"]["name"]
print(f"📦 Loading VLM model: {MODEL_NAME}")
print(f"   Using 4-bit quantization + FP16 for non-quantized parts")
print(f"   This will download ~5-15GB on first run...")
print()

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Free GPU memory before loading
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_properties(0).name} ({total_vram:.1f} GB)")
    free_vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    print(f"Free VRAM before loading: {free_vram:.1f} GB")

start = time.time()

# Load processor (handles images + text)
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True, max_pixels=1280*28*28)

# Load VLM model with 4-bit quantization
if device == "cuda":
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,       # CRITICAL: vision encoder + embeddings in FP16 (not FP32)
        device_map={"": 0},              # Load all on GPU 0, but let accelerate handle memory
        low_cpu_mem_usage=True,            # Reduce peak memory during loading
        trust_remote_code=True,
    )
else:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
    model = model.to(device)

model.eval()

elapsed = time.time() - start
vram_used = torch.cuda.memory_allocated() / 1e9 if device == "cuda" else 0

print(f"\n✅ VLM loaded in {elapsed:.0f}s: {MODEL_NAME}")
if device == "cuda":
    print(f"   VRAM used: {vram_used:.1f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   Ready for extraction!")


## Step 5: Vision Model Ready

The **Qwen 2.5 VL** model is already loaded (Step 4). It handles both vision and text.
**No separate OCR step needed** \u2014 the model sees the document image directly.


In [ ]:
# Vision-Language model info
print(f"👁\ufe0f  VLM Model: {MODEL_NAME}")
print(f"   Processor: {processor.__class__.__name__}")
print(f"   The model can see images AND understand text.")
print(f"   No EasyOCR or separate OCR step needed!")
print(f"\n   Architecture: Image Encoder + Language Decoder")
print(f"   Input: Document image (PNG/JPEG) + Text prompt")
print(f"   Output: Structured JSON extraction")


## Step 6: Define Extraction Pipeline (VLM \u2014 Vision + Language)

**VLM pipeline (much simpler than OCR):**
1. **PDF \u2192 Images** \u2014 Convert PDF pages to high-res images (PyMuPDF, 300 DPI)
2. **VLM Extraction** \u2014 Qwen 2.5 VL sees the image + prompt \u2192 outputs JSON
3. **Robust JSON parser** \u2014 auto-fix trailing commas, single quotes from LLM output
4. **Smart validator** \u2014 cross-check totals, amount_in_words validation

**No OCR, no preprocessing, no table formatting needed!** The VLM does it all.


In [ ]:
import numpy as np
import re
import fitz  # PyMuPDF
from PIL import Image
import io
import base64


# ======================================================================
# PDF/Image to array conversion
# ======================================================================

def document_to_images(file_bytes: bytes) -> list:
    """Convert PDF bytes to a list of page images (numpy arrays)."""
    images = []
    doc = fitz.open(stream=file_bytes, filetype="pdf")
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img_data = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_data)).convert("RGB")
        images.append(np.array(img))
    doc.close()
    return images


def image_to_numpy(file_bytes: bytes) -> np.ndarray:
    """Convert an image file to a numpy array."""
    img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    return np.array(img)


def image_to_base64(image: np.ndarray) -> str:
    """Convert a numpy array image to base64 string."""
    img = Image.fromarray(image)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# ======================================================================
# French number words parser
# ======================================================================

FRENCH_UNITS = {
    "zero": 0, "un": 1, "une": 1, "deux": 2, "trois": 3, "quatre": 4,
    "cinq": 5, "six": 6, "sept": 7, "huit": 8, "neuf": 9, "dix": 10,
    "onze": 11, "douze": 12, "treize": 13, "quatorze": 14, "quinze": 15,
    "seize": 16, "dix-sept": 17, "dix-huit": 18, "dix-neuf": 19,
    "vingt": 20, "trente": 30, "quarante": 40, "cinquante": 50,
    "soixante": 60, "soixante-dix": 70, "soixante-et-onze": 71,
    "quatre-vingts": 80, "quatre-vingt": 80, "quatre-vingt-un": 81,
    "quatre-vingt-onze": 91, "quatre-vingt-dix": 90,
}
FRENCH_MULTIPLIERS = {
    "cent": 100, "cents": 100, "mille": 1000,
    "million": 1000000, "millions": 1000000,
}


def french_words_to_number(text: str):
    """Convert French number words to a numeric value."""
    if not text:
        return None
    text = text.lower().strip().replace(" et ", " ")
    for stop in ["euros", "dirhams", "dhs", "dh"]:
        if stop in text:
            text = text.split(stop)[0].strip()
    decimal_value = 0.0
    if "virgule" in text:
        parts = text.split("virgule", 1)
        text = parts[0].strip()
        decimal_part = parts[1].strip().split()
        if decimal_part:
            dec_str = ""
            for d in decimal_part:
                for name, val in FRENCH_UNITS.items():
                    if d.strip() == name:
                        dec_str += str(val)
                        break
            if dec_str:
                decimal_value = int(dec_str) / (10 ** len(dec_str))
    tokens = re.split(r"[\s-]+", text)
    tokens = [t for t in tokens if t]
    result, current = 0, 0
    for tok in tokens:
        try:
            current += int(float(tok))
            continue
        except ValueError:
            pass
        if tok in FRENCH_MULTIPLIERS:
            mult = FRENCH_MULTIPLIERS[tok]
            if mult >= 1000:
                if current == 0:
                    current = 1
                result += current * mult
                current = 0
            elif mult == 100:
                current = current * 100 if current else 100
            continue
        if tok in FRENCH_UNITS:
            current += FRENCH_UNITS[tok]
    result += current + decimal_value
    return result if result > 0 else None

# ======================================================================
# Robust JSON parser
# ======================================================================

def fix_json_string(json_str):
    s = json_str.strip()
    # NOTE: Do NOT strip # or // as comments - they appear in JSON string values (e.g. "# 427, Type Zone...")
    if '"' not in s and "'" in s:
        s = s.replace("'", '"')
    s = re.sub(r',(\s*[\]}])', r'\1', s)
    return s


# ======================================================================
# Clean and validate
# ======================================================================

def clean_llm_output(data):
    def clean_value(val):
        if isinstance(val, str):
            val = val.replace("[low confidence]", "").strip()
            # Fix string "null" -> actual null
            if val.lower() == "null":
                return None
            return val
        return val
    def clean_recursive(obj):
        if isinstance(obj, dict):
            return {k: clean_recursive(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [clean_recursive(item) for item in obj]
        elif isinstance(obj, str):
            return clean_value(obj)
        return obj
    return clean_recursive(data)


def fix_unit_field_swap(data):
    warnings = []
    items = data.get("order", {}).get("items", [])
    for item in items:
        unit = item.get("unit")
        if unit is not None and isinstance(unit, (int, float)):
            item["unit_price_ht"] = float(unit)
            item["unit"] = None
            warnings.append(f"Swapped numeric unit ({unit}) to unit_price_ht for ref {item.get('reference', '?')}")
        elif unit is not None and isinstance(unit, str):
            try:
                num = float(unit.strip().replace(" ", ""))
                item["unit_price_ht"] = num
                item["unit"] = None
                warnings.append(f"Swapped numeric unit string to unit_price_ht for ref {item.get('reference', '?')}")
            except ValueError:
                pass
    return warnings


def validate_extraction(data):
    warnings = []
    MAX_RATIO = 3.0
    order = data.get("order", {})
    items = order.get("items", [])
    totals = order.get("totals", {})
    swap_warnings = fix_unit_field_swap(data)
    warnings.extend(swap_warnings)
    # Validate individual line items: qty * unit_price should ≈ total_line_ht
    for item in items:
        qty = item.get("quantity")
        price = item.get("unit_price_ht")
        total = item.get("total_line_ht")
        if qty and price and total and qty > 0 and price > 0:
            computed = round(qty * price, 2)
            ratio = computed / total if total != 0 else float("inf")
            if ratio > 100:  # qty likely has French comma decimal error (e.g. 64000 instead of 64)
                corrected_qty = round(total / price, 4)
                item["quantity"] = corrected_qty
                warnings.append(f"Corrected quantity {qty} -> {corrected_qty} (qty*price={computed} vs total={total})")
            elif ratio < 0.01:  # qty might be too small
                pass  # could be division issue, skip
    for item in items:
        qty = item.get("quantity")
        line_total = item.get("total_line_ht")
        unit_price = item.get("unit_price_ht")
        if (unit_price is None or unit_price == 0) and qty and line_total and qty > 0:
            computed = round(line_total / qty, 2)
            item["unit_price_ht"] = computed
            warnings.append(f"Computed unit_price_ht={computed} for ref {item.get('reference', '?')}")
    if items:
        computed_gross = round(sum(item.get("total_line_ht", 0) or 0 for item in items), 2)
        declared_gross = totals.get("gross_amount_ht")
        if declared_gross is not None and declared_gross > 0:
            ratio = computed_gross / declared_gross if declared_gross != 0 else float('inf')
            if ratio < (1/MAX_RATIO) or ratio > MAX_RATIO:
                warnings.append(f"Items sum ({computed_gross}) vs gross ({declared_gross}) differ by {ratio:.1f}x - keeping declared")
            elif abs(computed_gross - declared_gross) > 0.02:
                totals["gross_amount_ht"] = computed_gross
        elif declared_gross is None:
            totals["gross_amount_ht"] = computed_gross
    gross = totals.get("gross_amount_ht") or 0
    discount = totals.get("discount", 0) or 0
    declared_net = totals.get("net_amount_ht")
    expected_net = round(gross - discount, 2)
    if declared_net is not None and declared_net > 0:
        ratio = declared_net / expected_net if expected_net != 0 else float('inf')
        if ratio < (1/MAX_RATIO) or ratio > MAX_RATIO:
            warnings.append(f"Net HT ({declared_net}) vs computed ({expected_net}) differ by {ratio:.1f}x - keeping declared")
        elif abs(declared_net - expected_net) > 0.02:
            totals["net_amount_ht"] = expected_net
    else:
        totals["net_amount_ht"] = expected_net
    net_ht = totals.get("net_amount_ht") or 0
    vat_rate = totals.get("vat_rate")
    declared_vat = totals.get("vat_amount")
    if net_ht and vat_rate:
        computed_vat = round(net_ht * vat_rate / 100, 2)
        if declared_vat is not None and declared_vat > 0:
            ratio = declared_vat / computed_vat if computed_vat != 0 else float('inf')
            if ratio < (1/MAX_RATIO) or ratio > MAX_RATIO:
                warnings.append(f"VAT ({declared_vat}) vs computed ({computed_vat}) differ by {ratio:.1f}x - keeping declared")
            elif abs(computed_vat - declared_vat) > 0.02:
                totals["vat_amount"] = computed_vat
        elif declared_vat is None:
            totals["vat_amount"] = computed_vat
    vat = totals.get("vat_amount", 0) or 0
    computed_ttc = round(net_ht + vat, 2)
    declared_ttc = totals.get("net_amount_ttc")
    if declared_ttc is not None and declared_ttc > 0:
        if abs(computed_ttc - declared_ttc) > 0.02:
            # TTC PRIORITY: trust the document's explicit TTC, compute implied VAT
            implied_vat = round(declared_ttc - net_ht, 2)
            if abs(implied_vat - vat) > 0.02:
                totals["vat_amount"] = implied_vat
                implied_rate = round(implied_vat / net_ht * 100, 2) if net_ht > 0 else 0
                if abs(implied_rate - (totals.get("vat_rate") or 0)) > 0.02:
                    totals["vat_rate"] = implied_rate
                warnings.append(f"TTC doc={declared_ttc} used. Implied VAT={implied_vat} (rate={implied_rate}%) instead of declared VAT={vat}")
    elif declared_ttc is None:
        totals["net_amount_ttc"] = computed_ttc
    amount_words = totals.get("amount_in_words")
    if amount_words and isinstance(amount_words, str):
        words_value = french_words_to_number(amount_words)
        if words_value and words_value > 0:
            current_ttc = totals.get("net_amount_ttc") or 0
            if current_ttc > 0:
                ratio = current_ttc / words_value if words_value != 0 else float('inf')
                if ratio < 0.67 or ratio > 1.5:
                    warnings.append(f"amount_in_words={words_value} vs TTC={current_ttc} differ by {ratio:.1f}x - using words")
                    scale = words_value / current_ttc
                    for field in ["net_amount_ttc", "vat_amount", "net_amount_ht", "gross_amount_ht"]:
                        old = totals.get(field)
                        if old and isinstance(old, (int, float)) and old > 0:
                            totals[field] = round(old * scale, 2)
    if warnings:
        data["_validation_warnings"] = warnings
        print(f"   {len(warnings)} validation issue(s)")
        for w in warnings:
            print(f"      - {w}")
    else:
        print(f"   All totals validated")
    return data



# ======================================================================
# Post-processing fixes
# ======================================================================

def fix_country_names(data):
    """Fix common country name OCR errors."""
    country_fixes = {
        "moroc": "Maroc",
        "maroc": "Maroc",
        "morocco": "Maroc",
        "marocc": "Maroc",
    }
    for section in ["seller", "buyer"]:
        country = data.get(section, {}).get("country")
        if country and isinstance(country, str):
            fixed = country_fixes.get(country.strip().lower())
            if fixed:
                data[section]["country"] = fixed
    return data



# ======================================================================
# VLM Extraction
# ======================================================================

def run_vlm_extraction(image):
    # Resize image to max 1344px on longest side (prevents OOM on T4)
    pil_image = Image.fromarray(image)
    max_dim = 1344
    w, h = pil_image.size
    if max(w, h) > max_dim:
        scale = max_dim / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        pil_image = pil_image.resize((new_w, new_h), Image.LANCZOS)
    extraction_hints = ""
    prompt = get_prompt_for_language(current_lang)
    try:
        user_prompt = prompt.format(
            extraction_hints=extraction_hints,
            output_schema=json.dumps(output_schema, indent=2),
        )
    except KeyError:
        user_prompt = prompt.replace("{extraction_hints}", extraction_hints)
        user_prompt = user_prompt.replace("{output_schema}", json.dumps(output_schema, indent=2))
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": user_prompt},
            ]
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.15,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    generated_ids = outputs[:, inputs.input_ids.shape[-1]:]
    raw_output = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    # Strip markdown code fences if model wrapped JSON in them
    raw_output = re.sub(r'^```json\s*', '', raw_output.strip())
    raw_output = re.sub(r'\s*```$', '', raw_output.strip())
    return raw_output


# ======================================================================
# Main extraction pipeline
# ======================================================================

def extract_from_file(file_bytes, filename):
    global current_lang
    print(f"\n  Processing: {filename}")
    ext = os.path.splitext(filename)[1].lower()
    current_lang = detect_language(file_bytes, filename)
    lang_label = "\U0001f1eb\U0001f1f7 French" if current_lang == "fr" else "\U0001f1ec\U0001f1e7 English"
    print(f"    Language: {lang_label}")
    if ext == ".pdf":
        images = document_to_images(file_bytes)
    else:
        images = [image_to_numpy(file_bytes)]
    print(f"    Pages: {len(images)}")
    all_results = []
    for idx, img in enumerate(images):
        print(f"    VLM page {idx+1}/{len(images)}...", end=" ", flush=True)
        raw_output = run_vlm_extraction(img)
        print(f"{len(raw_output)} chars")
        json_match = re.search(r'\{[\s\S]*\}', raw_output)
        if json_match:
            raw_json = fix_json_string(json_match.group())
            try:
                data = json.loads(raw_json)
            except json.JSONDecodeError:
                flat = re.sub(r'\s+', ' ', raw_json)
                try:
                    data = json.loads(flat)
                except json.JSONDecodeError:
                    data = {"raw_output": raw_output[:2000], "parse_error": True}
        else:
            data = {"raw_output": raw_output[:2000], "parse_error": True}
        all_results.append(data)
    data = all_results[0]
    if isinstance(data, dict) and "output" in data and isinstance(data["output"], dict):
        inner = data["output"]
        if "seller" in inner or "buyer" in inner or "order" in inner:
            print(f"    Unwrapping output schema wrapper")
            data = inner
    data = clean_llm_output(data)
    data = fix_country_names(data)
    data = validate_extraction(data)
    if not data or data == {}:
        data = {"parse_error": True, "raw_output": raw_output[:2000], "_debug": {"pages": len(images)}}
    print(f"    Done! Keys: {list(data.keys())[:10]}")
    return data


print("VLM extraction pipeline ready!")


## Step 7: Quick Test

Test the VLM pipeline with a synthetic French purchase order.
The model will SEE the PDF image and extract data directly.


In [ ]:
# Create a test PDF with synthetic French PO text
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import mm

def create_test_pdf():
    """Create a synthetic French purchase order PDF for testing."""
    buf = io.BytesIO()
    c = canvas.Canvas(buf, pagesize=A4)
    w, h = A4
    
    # Header
    c.setFont("Helvetica-Bold", 18)
    c.drawCentredString(w/2, h - 40*mm, "BON DE COMMANDE")
    
    c.setFont("Helvetica", 10)
    c.drawString(30*mm, h - 60*mm, "N\u00b0 BC-2025-0042")
    c.drawString(100*mm, h - 60*mm, "Date: 15/04/2025")
    c.drawString(30*mm, h - 68*mm, "FERME")
    
    # Seller
    c.setFont("Helvetica-Bold", 10)
    c.drawString(30*mm, h - 85*mm, "Fournisseur:")
    c.setFont("Helvetica", 9)
    c.drawString(55*mm, h - 85*mm, "Martin Import SARL")
    c.drawString(55*mm, h - 91*mm, "45 Avenue des Champs, 69003 Lyon")
    c.drawString(55*mm, h - 97*mm, "SIRET: 123 456 789 00012")
    c.drawString(55*mm, h - 103*mm, "TVA: FR 12 345678901")
    
    # Buyer
    c.setFont("Helvetica-Bold", 10)
    c.drawString(120*mm, h - 85*mm, "Client:")
    c.setFont("Helvetica", 9)
    c.drawString(135*mm, h - 85*mm, "BeeOne SAS")
    c.drawString(135*mm, h - 91*mm, "12 Rue de la Paix, 75002 Paris")
    c.drawString(135*mm, h - 97*mm, "Contact: Jean Dupont")
    c.drawString(135*mm, h - 103*mm, "T\u00e9l: 01 42 33 44 55")
    
    # Table header
    y = h - 130*mm
    c.setFont("Helvetica-Bold", 8)
    headers = ["R\u00e9f", "D\u00e9signation", "Qt\u00e9", "Unit\u00e9", "Prix unit. HT", "Total HT"]
    x_positions = [30*mm, 50*mm, 110*mm, 125*mm, 140*mm, 165*mm]
    for hdr, x in zip(headers, x_positions):
        c.drawString(x, y, hdr)
    
    # Table lines
    c.setFont("Helvetica", 8)
    items = [
        ("A001", "Bureau ergonomique adjustable", "5", "pcs", "450,00", "2 250,00"),
        ("A002", "Chaise de bureau avec accoudoirs", "10", "pcs", "150,00", "1 500,00"),
        ("B010", "Ecran 27 pouces 4K IPS", "3", "pcs", "320,00", "960,00"),
        ("C005", "Clavier m\u00e9canique sans fil", "10", "pcs", "89,00", "890,00"),
    ]
    
    for i, item in enumerate(items):
        y -= 7*mm
        for val, x in zip(item, x_positions):
            c.drawString(x, y, val)
    
    # Totals
    y -= 12*mm
    c.setFont("Helvetica", 8)
    c.drawString(140*mm, y, "Total HT:")
    c.drawString(165*mm, y, "5 600,00")
    
    y -= 6*mm
    c.drawString(140*mm, y, "Remise 5%:")
    c.drawString(165*mm, y, "-280,00")
    
    y -= 6*mm
    c.drawString(140*mm, y, "Net HT:")
    c.drawString(165*mm, y, "5 320,00")
    
    y -= 6*mm
    c.drawString(140*mm, y, "TVA 20%:")
    c.drawString(165*mm, y, "1 064,00")
    
    y -= 6*mm
    c.setFont("Helvetica-Bold", 9)
    c.drawString(140*mm, y, "Total TTC:")
    c.drawString(165*mm, y, "6 384,00")
    
    # Footer
    y -= 15*mm
    c.setFont("Helvetica", 7)
    c.drawString(30*mm, y, "D\u00e9lai de livraison: 15 jours")
    c.drawString(30*mm, y - 5*mm, "Conditions de paiement: 30 jours fin de mois")
    c.drawString(30*mm, y - 10*mm, "Montant TTC en lettres: six mille trois cent quatre-vingt-quatre euros")
    
    c.save()
    buf.seek(0)
    return buf.getvalue()


# Create and test
test_pdf = create_test_pdf()
result = extract_from_file(test_pdf, "test_po.pdf")

print("\n" + "="*60)
print("EXTRACTION RESULT")
print("="*60)
print(json.dumps(result, indent=2, ensure_ascii=False))

## Step 8: Batch Extraction UI (Gradio)

Upload **multiple documents at once** and extract all of them in batch.
The UI lets you:
- Upload multiple PDF or image files simultaneously
- See progress and results for each document
- View per-document summary and full JSON
- Download all results as a single JSON file
- Toggle between single-file and batch mode

In [ ]:
import gradio as gr
import time as time_mod


def format_single_summary(result, filename, idx):
    if isinstance(result, dict) and "output" in result and isinstance(result["output"], dict):
        if "seller" in result["output"] or "order" in result["output"]:
            result = result["output"]

    seller = result.get("seller", {})
    buyer = result.get("buyer", {})
    order = result.get("order", {})
    totals = order.get("totals", {})
    items = order.get("items", [])

    status = result.get("parse_error") and "\u274c Parse Error" or "\u2705 Extracted"

    lines = [
        f"### Document {idx}: `{filename}` {status}\n",
        f"**Order:** {order.get('order_number', 'N/A')} | ",
        f"**Date:** {order.get('date', 'N/A')} | ",
        f"**Ferme:** {order.get('ferme', 'N/A')}  ",
        f"| **Seller:** {seller.get('name', 'N/A')}  ",
        f"| **Buyer:** {buyer.get('company_name', 'N/A')}  ",
        f"| **Items:** {len(items)} line(s)  ",
        f"| **Total TTC:** {totals.get('net_amount_ttc', 'N/A')}\n",
    ]

    add_info = result.get("additional_information")
    if not add_info:
        add_info = totals.get("additional_information")
    if add_info:
        lines.append(f"**Additional Info:** {add_info}\n")

    if items:
        lines.append(f"| # | Reference | Designation | Qty | Unit | Price HT | Total HT |\n")
        lines.append(f"|---|-----------|-------------|-----|------|----------|----------|\n")
        for i, item in enumerate(items, 1):
            lines.append(
                f"| {i} | {item.get('reference', '')} | {item.get('designation', '')} | "
                f"{item.get('quantity', '')} | {item.get('unit', '')} | "
                f"{item.get('unit_price_ht', '')} | {item.get('total_line_ht', '')} |\n"
            )
        lines.append("")

    if result.get("_validation_warnings"):
        lines.append("**Validation warnings:**\n")
        for w in result["_validation_warnings"]:
            lines.append(f"- {w}\n")

    return "".join(lines)


def extract_single(file):
    if file is None:
        return "Please upload a document.", "", ""
    file_bytes = open(file, "rb").read()
    filename = os.path.basename(file)
    result = extract_from_file(file_bytes, filename)

    summary = format_single_summary(result, filename, 1)
    json_output = json.dumps(result, indent=2, ensure_ascii=False)
    vlm_info = f"VLM: {MODEL_NAME}\nNo separate OCR needed \u2014 model reads document image directly."
    return summary, json_output, vlm_info


def extract_batch(files):
    if not files:
        return "Please upload at least one document.", "", ""

    all_results = {}
    all_summaries = []
    total_start = time_mod.time()

    for idx, file in enumerate(files, 1):
        filename = os.path.basename(file)
        file_bytes = open(file, "rb").read()

        try:
            start = time_mod.time()
            result = extract_from_file(file_bytes, filename)
            elapsed = time_mod.time() - start
            all_results[filename] = result
            summary = format_single_summary(result, filename, idx)
            all_summaries.append(f"<!-- {idx} -->\n{summary}")
        except Exception as e:
            import traceback
            all_results[filename] = {"parse_error": True, "error": str(e)}
            all_summaries.append(
                f"### Document {idx}: `{filename}` \u274c Error\n\n"
                f"`{e}`\n"
            )

    total_elapsed = time_mod.time() - total_start
    header = (
        f"## Batch Extraction Results\n\n"
        f"**Documents processed:** {len(files)} | "
        f"**Total time:** {total_elapsed:.1f}s | "
        f"**Model:** {MODEL_NAME}\n\n"
        f"---\n\n"
    )
    summary_output = header + "\n".join(all_summaries)
    json_output = json.dumps(all_results, indent=2, ensure_ascii=False)

    vlm_info = (
        f"VLM: {MODEL_NAME}\n"
        f"Documents: {len(files)}\n"
        f"Total time: {total_elapsed:.1f}s\n"
        f"No separate OCR needed \u2014 model reads each document image directly."
    )

    return summary_output, json_output, vlm_info


with gr.Blocks(title="BeeOne PO Extractor") as demo:
    gr.Markdown("# \U0001f4e6 BeeOne Purchase Order Extractor")
    gr.Markdown(
        "Upload purchase orders (**PDF or image**) \u2014 supports **French** and **English**. "
        "AI extracts structured JSON automatically.\n\n"
        "**Single mode**: one document at a time. "
        "**Batch mode**: upload multiple documents and extract all at once."
    )

    with gr.Tabs():
        with gr.TabItem("\U0001f4c2 Single File"):
            with gr.Row():
                single_file = gr.File(
                    label="Upload Document (PDF or Image)",
                    file_types=[".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"],
                )
            single_btn = gr.Button("\U0001f50d Extract", variant="primary", size="lg")

            with gr.Tabs():
                with gr.TabItem("Summary"):
                    single_summary = gr.Markdown()
                with gr.TabItem("JSON Output"):
                    single_json = gr.Code(label="Extracted JSON", language="json")
                with gr.TabItem("Debug"):
                    single_debug = gr.Code(label="VLM Debug Info", language=None)

            single_btn.click(
                fn=extract_single,
                inputs=[single_file],
                outputs=[single_summary, single_json, single_debug],
            )

        with gr.TabItem("\U0001f4c1 Batch (Multiple Files)"):
            batch_files = gr.File(
                label="Upload Multiple Documents",
                file_types=[".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"],
                file_count="multiple",
            )
            batch_btn = gr.Button("\u26a1 Extract All", variant="primary", size="lg")

            with gr.Tabs():
                with gr.TabItem("All Summaries"):
                    batch_summary = gr.Markdown()
                with gr.TabItem("All JSON"):
                    batch_json = gr.Code(label="All Results (JSON)", language="json")
                with gr.TabItem("Debug"):
                    batch_debug = gr.Code(label="Batch Debug Info", language=None)

            batch_btn.click(
                fn=extract_batch,
                inputs=[batch_files],
                outputs=[batch_summary, batch_json, batch_debug],
            )


demo.launch(share=True, debug=False)
print("\u2705 Gradio UI launched! Click the public URL above.")
